# 01 · Dataset exploration

## The story (explained like you're five)
Some people had a **stroke**, which can make one hand hard to move. Doctors asked 50 of them to **imagine squeezing** their left or right hand while a stretchy cap with 29 little sensors listened to the electricity in their head. That recording is called **EEG**.

Our job: look at the wiggles and guess **which hand** they were thinking about.

Today we only **look** at the data. No guessing yet.

**Facts from the project brief:** 50 acute-stroke patients, 29 electrodes, 40 trials each (20 left + 20 right), 4 seconds per trial, recorded at 500 Hz (we shrink to 250 Hz).

> ⚠️ The first download can be big and slow. Start with 5 patients.

In [ ]:
# --- Setup (works on your laptop AND on Google Colab) ---
import os, sys
if "google.colab" in sys.modules and not os.path.exists("src"):
    # Colab: get the project first, e.g.  !git clone <your-repo-url> && %cd stroke-rehab-bci/notebooks
    pass
sys.path.append("..")          # so we can "import src"
# !pip install -q moabb mne scikit-learn pandas matplotlib   # uncomment on Colab
import numpy as np, pandas as pd, matplotlib.pyplot as plt


In [ ]:
from src.preprocessing import load_liu2024
X, y, metadata = load_liu2024(subjects=[1, 2, 3, 4, 5])   # use None later for all 50
print("X shape:", X.shape, " -> (trials, channels, time points)")
print("Expected about (200, 29, 1000) for 5 patients. If it differs, find out why before moving on!")

### What are `X`, `y`, `metadata`?
- `X` = the EEG trials (the wiggles)
- `y` = the answer for each trial (left or right)
- `metadata` = which patient each trial came from

In [ ]:
print(y[:10])
print(metadata.head())
print(pd.Series(y).value_counts())
print(metadata["subject"].value_counts().sort_index())

### What does 500 Hz mean?
The computer measured the brain **500 times every second**. 4 seconds → ~2000 measurements per sensor. We shrink to 250 Hz (~1000 measurements) to keep things light. That's fine because we only care about wiggles up to 30 Hz.

In [ ]:
t = np.arange(X.shape[2]) / 250
fig, axes = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
for ax, ch in zip(axes, [0, 1, 2]):
    ax.plot(t, X[0, ch]); ax.set_ylabel(f"channel {ch}")
axes[-1].set_xlabel("seconds"); axes[0].set_title(f"One trial, label = {y[0]}")
plt.tight_layout(); os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/eeg_example.png", dpi=150)